<a href="https://colab.research.google.com/github/DABMASTER-Brought-me-into-this/NeuroLearn/blob/main/SlideMDIOWordClassifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [98]:
# Standard Imports
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [99]:
### HYPERPARAMETERS
MAX_WORD_LEN = 30
BN_EMA_SF = 0.95
LR = 0.01
N_EMBD = 10
NUM_OF_NEURONS_L1 = 128
BATCH_SIZE = 32
NUM_OF_NEURONS_L2 = 64
NUM_OF_NEURONS_L3 = 16

In [100]:
# "Opening" the Dataset
# To Be Replaced w/ real dataset & preprocessing
df = pd.read_csv("slidemdtestdata.csv")

In [101]:
# String to Integer Index
stoi = {chr(i): i - ord('a') + 1 for i in range(ord('a'), ord('z')+1)}
stoi['.'] = 0 # Padding Char
itos = {ix: char for char, ix in stoi.items()}

In [102]:
# Encoding the Words
def tok_word(word):
  # Array of chars
  letters = list(word)
  enc_letter = lambda letter: stoi[letter] if letter in stoi.keys() else 0
  enc_word = list(map(enc_letter, letters))

  enc_word_len = len(enc_word)
  if enc_word_len < MAX_WORD_LEN:
    enc_word.extend([0] * (MAX_WORD_LEN - enc_word_len))

  return enc_word

In [103]:
# Tokenizing Words in Dataset
inputs = df['word'].tolist()
X = np.array(list(map(tok_word, inputs)))
Y = np.array(df['is_medical']).reshape(-1, 1)

In [104]:
### Classes for Neural Network Micrograd Style
class Embedding:
  def __init__ (self, size, n_embd):
    self.n_embd = n_embd
    self.C = np.random.randn(*size, n_embd)
    self.dC = np.zeros_like(self.C)

  def __call__ (self, X):
    self.X = X
    self.out = self.C[self.X]
    return self.out

  def __backwards(self, dup):
    np.add.at(self.dC, self.X.ravel(), dup.reshape(-1, self.n_embd))
    return None

  def __parameters(self):
    return [(self.C, self.dC)]

  def __wipegrad(self):
    self.dC = np.zeros_like(self.C)

class Reshape:
  def __init__ (self, shape):
    self.shape = shape

  def __call__ (self, X):
    self.X = X
    self.out = self.X.reshape(self.shape)
    return self.out

  def __backwards(self, dup):
    self.dout = dup.reshape(self.X.shape)
    return self.dout

  def __parameters(self):
    return []

  def __wipegrad(self):
    pass

class Linear:
  def __init__ (self, size):
    self.W = np.random.randn(*size) * 0.01
    self.B = np.zeros(size[-1])
    self.dW = np.zeros_like(self.W)
    self.dB = np.zeros_like(self.B)

  def __call__ (self, X):
    self.X = X
    self.out = self.X @ self.W + self.B
    return self.out

  def __backwards (self, dup):
    self.dW += X @ dup.T
    self.dB += dup.sum(axis = (tuple(range(self.B.ndim - 1))))
    self.dout = dup.T @ self.W.T

  def __parameters(self):
    return [(self.W, self.dW), (self.B, self.dB)]

  def __wipegrad (self):
    self.dW = np.zeros_like(self.W)
    self.dB = np.zeros_like(self.B)

class BatchNorm:
  def __init__(self, size, training=True):
    self.training = training
    self.g = np.ones((size))
    self.b = np.zeros((size))
    self.dg = np.zeros_like(self.g)
    self.db = np.zeros_like(self.b)
    self.running_mean = None
    self.running_var = None

  def __call__ (self, X):
    if self.training:
      # Calculate New Stats
      self.xmean = np.mean(X, axis = 0, keepdims = True)
      self.xvar = np.var(X, axis = 0, keepdims = True)

      # Assign First Run to First Means/Vars
      if self.running_mean is None and self.running_var is None:
        self.running_mean = self.xmean
        self.running_var = self.xvar
      else:
        self.running_mean = BN_EMA_SF * self.running_mean + (1 - BN_EMA_SF) * self.xmean
        self.running_var = BN_EMA_SF * self.running_var + (1 - BN_EMA_SF) * self.xvar
    else:
      self.xmean = self.running_mean
      self.xvar = self.running_var

    self.X = X # Saving Input for BP
    # Batch Norm
    self.raw = (self.X - self.xmean)/np.sqrt(self.xvar + 1e-5)
    self.out = self.g * self.raw + self.b
    return self.out

  def __backwards(self, dup):
    self.dg = np.sum(self.X * dup, axis = tuple(range(-1, -self.g.ndim - 1, -1)), keepdims = True)
    self.db = np.sum(dup, axis = tuple(range(-1, -self.b.ndim - 1, -1)), keepdims = True)
    self.dout = ((dup * self.dg) - (dup * self.dg).mean(axis=1, keepdims=True) - self.raw * ((dup * self.dg) * self.raw).mean(axis=1, keepdims=True))/(np.sqrt(self.xvar + 1e-5))
    return self.dout

  def __parameters(self):
    return [(self.g, self.dg), (self.b, self.db)]

  def __wipegrad(self):
    self.dg = np.zeros_like(self.g)
    self.db = np.zeros_like(self.b)

class LeakyReLu:
  def __call__ (self, X):
    self.X = X # saved for bp
    self.out = np.maximum(0.01 * self.X, self.X)
    return self.out

  def __backwards(self, dup):
    bg_deriv = np.ones_like(dup) * 0.01
    self.dout = np.where(self.X >= 0, dup, bg_deriv)
    return self.dout

  def __parameters(self):
    return []

  def __wipegrad(self):
    pass

class Sigmoid:
  def __call__ (self, X):
    self.X = X # svd for bp
    self.out = (1 + np.e ** -(self.X)) ** -1
    return self.out

  def __backwards(self, dup):
    self.dout = (np.e ** self.X) * (1 + np.e ** -(self.X)) ** -2
    return self.dout

  def __parameters(self):
    return []

  def __wipegrad(self):
    pass

class Model:
  def __init__(self, layers, lr):
    self.layers = layers
    self.lr = lr

  def __call__(self, X):
    self.X = X
    self.out = self.X
    for layer in self.layers:
      self.out = layer(self.out)
    return self.out

  def backwards(self, dup):
    self.dout = dup
    for layer in self.layers:
      self.dout = layer.__backwards(self.dout)

  def update_parameters(self):
    parameters = []
    for layer in self.layers:
      parameters.extend(layer.__parameters)

    for p,g in parameters:
      p -= g * self.lr

  def wipegrad(self):
    for layer in self.layers:
      layer.__wipegrad()


In [105]:
# My model
layers = [Embedding((X.shape[0],), N_EMBD), Reshape((BATCH_SIZE, -1)),
          Linear((MAX_WORD_LEN * N_EMBD, NUM_OF_NEURONS_L1)), BatchNorm((NUM_OF_NEURONS_L1)), LeakyReLu(),
          Linear((NUM_OF_NEURONS_L1, NUM_OF_NEURONS_L2)), BatchNorm((NUM_OF_NEURONS_L2)), LeakyReLu(),
          Linear((NUM_OF_NEURONS_L2, NUM_OF_NEURONS_L3)), BatchNorm((NUM_OF_NEURONS_L3)), LeakyReLu(),
          Linear((NUM_OF_NEURONS_L3, 1)), Sigmoid()]

nn = Model(layers, LR)

In [106]:
out = nn(X[0:BATCH_SIZE])
loss = (-(Y[0:BATCH_SIZE] * np.log(out) + (1 - Y[0:BATCH_SIZE]) * np.log(1 - out))).mean()
loss

np.float64(0.6898098760985725)

In [107]:
dloss = 1/BATCH_SIZE * (-Y[0:BATCH_SIZE]/out + (1 - Y[0:BATCH_SIZE])/(1 - out))